# PERTEMUAN 8 — PRAKTIKUM PEMBELAJARAN MESIN
## **Bagaimana EDA dan preprocessing menentukan pilihan Hierarchical Clustering atau K-Means?**

**DAIS32W · 150 menit praktikum · Praktikum individual **

> **Tujuan:** Mahasiswa dapat: (a) membuktikan keterkaitan materi Pertemuan 1–7 dengan struktur clustering; (b) mengaudit apakah jarak dan model layak diuji; (c) membandingkan Ward dan K-Means dengan data, fitur dan k yang sebanding; (d) melakukan ablation dan mempertanggungjawabkan keputusan tanpa label; serta (e) menyusun portofolio satu notebook dengan tabel eksperimen dan profil cluster..

**Sebelum mulai**

1. Buka notebook di Jupyter/Google Colab/Kaggle; gunakan *Python 3* dengan `numpy`, `pandas`, `matplotlib`, `scipy`, dan `scikit-learn`.
2. Jalankan cell berurutan dari atas ke bawah. Tulis jawaban di setiap kotak **✍️ JAWABAN ANDA** (ubah sel Markdown dengan *double click*).
3. **Jangan mengubah label cultivar Wine menjadi fitur**, atau menggunakannya untuk memilih model. Cluster tidak mempunyai label benar untuk pelatihan.
4. Setiap tabel/grafik perlu diinterpretasikan. Tidak cukup hanya screenshot atau menulis “skor meningkat”.
5. **Pilih algoritma berdasarkan dua jenis bukti:** kelayakan awal dari EDA, dan hasil perbandingan yang menggunakan representasi/evaluasi setara. Hasilnya boleh berbeda dari contoh dosen bila ada penjelasan yang benar.

**Identitas:** NIM: `255314093` · Nama: `Natalia Margareta` · Tanggal: `21/10/2026`.

### Peta hubungan teori P1–P7 dengan praktikum P8

| Materi terdahulu | Apa yang diuji dalam praktikum ini? | Bukti yang harus muncul |
|:--|:--|:--|
| **P1 — Problem framing & tipe belajar** | Kapan mencari kelompok tanpa label? Apa tujuan analisis? | Rumusan tujuan + unit analisis |
| **P2 — Vektor, matriks, jarak, objektif** | Jarak Euclidean, centroid/inertia, Ward ΔSSE | Cocokkan contoh enam Wine dari teori |
| **P3 — Desain eksperimen & generalisasi** | Perbandingan adil, satu perubahan per run, tidak memilih atas label tersembunyi | Tabel ablation + keterbatasan |
| **P4 — EDA & kualitas data** | Tipe fitur, sebaran, missing, duplikasi, outlier, korelasi | Issue log berdasar angka |
| **P5 — Cleaning & preprocessing** | Imputasi, scaling, penanganan nilai ekstrem, dampak transformasi | Hasil raw vs StandardScaler dan data terganggu |
| **P6 — Feature selection & PCA** | Redundansi, korelasi, kehilangan informasi | Perbandingan full vs selection vs PCA |
| **P7 — UTS integrasi** | Audit keputusan, eksperimen tercatat, keterulangan | FINAL_DECISION + README GitHub |

*Catatan*: Split training/validation/test tidak otomatis diwajibkan untuk **eksplorasi unsupervised deskriptif** atas satu kumpulan data. Jika metode akan diterapkan atau diklaim menggeneralisasi ke data baru, preprocessing serta evaluasi keluar-sampel harus dirancang secara terpisah. Jangan mengira bahwa cluster selalu memiliki ground truth.

## P0 — Memulai tanpa mengubah konfigurasi algoritma
**JALANKAN:** cell import, versi pustaka, dan seed. Notebook memakai dataset bawaan scikit-learn; **tidak memerlukan internet maupun CSV eksternal**. Jika `ModuleNotFoundError`, instal library pada environment Jupyter/Colab yang sesuai, lalu mulai ulang kernel.

In [ ]:
import sys, warnings, time
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.cluster.hierarchy import linkage, dendrogram
from scipy.spatial.distance import pdist, squareform
from sklearn.datasets import load_wine, make_blobs, make_moons
from sklearn.cluster import AgglomerativeClustering, KMeans
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.impute import SimpleImputer
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score, davies_bouldin_score, adjusted_rand_score
import sklearn, scipy
from IPython.display import display

SEED = 2026
rng = np.random.default_rng(SEED)
OUTDIR = Path('P08_output')
OUTDIR.mkdir(exist_ok=True)
pd.set_option('display.max_columns', 15)
print('Python:', sys.version.split()[0], '| scikit-learn:', sklearn.__version__, '| SciPy:', scipy.__version__)
print('Random seed:', SEED, '| Output:', OUTDIR.resolve())

## P1 — Dari hitungan teori ke Python
**SUMBER NYATA:** `sklearn.datasets.load_wine()` (178 sampel, 13 fitur numerik). Pengelompokan **tidak memakai** `wine.target`. Contoh dari teori memilih indeks `[0,2,59,60,175,176]` dan dua fitur (`alcohol`, `color_intensity`).

**JALANKAN:** tampilkan enam observasi, jarak A–B, dan dua skenario kelompok S1/S2. Cocokkan dengan Bagian 6 materi teori.

In [ ]:
wine = load_wine(as_frame=True)
X_raw = wine.data.copy()   # HANYA fitur kimia; label cultivar wine.target tidak masuk
X_raw.index.name = 'row_id'
contoh_id = [0, 2, 59, 60, 175, 176]
contoh = X_raw.loc[contoh_id, ['alcohol', 'color_intensity']].copy()
contoh.index = list('ABCDEF')
print('Dataset:', X_raw.shape, '| Target tersedia untuk sumber, tetapi TIDAK dimasukkan ke clustering.')
display(contoh)
dist = squareform(pdist(contoh.to_numpy(), metric='euclidean'))
print('Jarak A-B =', round(dist[0,1],4), '(teori ≈ 1.0707)')
S1 = [0,0,1,1,2,2]   # A,B | C,D | E,F
S2 = [0,1,0,1,2,2]   # A,C | B,D | E,F
print('Silhouette S1 =', round(silhouette_score(contoh, S1),4), '(teori ≈ 0.686)')
print('Silhouette S2 =', round(silhouette_score(contoh, S2),4), '(teori ≈ 0.098)')
assert X_raw.shape == (178, 13)

**✍️ JAWABAN ANDA — P1.** Apa arti label `S1`/`S2`? Mengapa rata-rata silhouette dapat berbeda meski nilai keenam observasi sama? Jelaskan apakah hasil enam baris sudah bisa menjadi kesimpulan untuk seluruh Wine Dataset.

**Jawab di sini:** `
- Arti label S1 dan S2, keduanya adalah dua skenario pengelompokan (vektor label cluster) untuk enam titik yang sama, yaitu A sampai F. Keduanya berasal dari contoh teori, bukan hasil model dan bukan label kelas.

S1 = {A,B} | {C,D} | {E,F}: titik yang berdekatan digabung.
S2 = {A,C} | {B,D} | {E,F}: A dipasangkan dengan C, dan B dengan D, padahal pasangan itu berjauhan.

- Mengapa rata-rata observasi sama? Silhouette dihitung dari partisi, bukan hanya dari titiknya. Untuk tiap titik, a(i) adalah jarak rata-rata ke anggota clusternya sendiri, dan b(i) adalah jarak rata-rata ke cluster tetangga terdekat. Nilai silhouette-nya (b − a) / max(a, b).

Pada S1, pasangan A–B berjarak 1,0707 (dekat), jadi a(i) kecil dan b(i) besar. Silhouette-nya 0,6864 dan pada S2, A dipasangkan dengan C yang berjarak sekitar 4,13 (jauh), jadi a(i) membesar dan b(i) mengecil. Silhouette-nya hanya 0,0979.

Jadi silhouette menilai kualitas pengelompokan, bukan kualitas datanya

- Belum, karena:

1. Enam dari 178 sampel hanya 3,4% dari data.
2. Keenamnya dipilih manual, bukan acak.
3. Hanya 2 dari 13 fitur yang dipakai (alcohol dan color_intensity), dan belum distandardisasi.
Dengan n sebesar enam, silhouette mudah berubah drastis hanya karena satu titik berpindah cluster.

Contoh ini hanya membuktikan bahwa rumus jarak dan silhouette di Python cocok dengan hitungan teori (1,0707; 0,686; 0,098). Ia tidak membuktikan adanya struktur cluster pada Wine. Itu baru bisa dinilai lewat EDA dan perbandingan eksperimen pada seluruh 178 sampel dan 13 fitur (P2-P4).`

## P2 — EDA sebelum clustering: apakah metode berbasis jarak masuk akal?

> **Pertanyaan keputusan:** dataset berukuran berapa, fitur apa yang tersedia, adakah kualitas data bermasalah, apakah skala timpang, apakah banyak fitur redundan, dan apakah perlu mengeksplorasi hierarki?

### P2-A. Profil data dan audit missing/duplikasi
**JALANKAN:** lihat `head`, ukuran, tipe data, missing, duplikasi, rentang, dan sebaran. **Jangan langsung scaling / clustering**.

In [ ]:
display(X_raw.head(5))
report_eda = pd.DataFrame({
    'dtype':X_raw.dtypes.astype(str),
    'missing':X_raw.isna().sum(),
    'mean':X_raw.mean(),
    'std':X_raw.std(ddof=0),
    'min':X_raw.min(),
    'median':X_raw.median(),
    'max':X_raw.max(),
})
report_eda['range'] = report_eda['max'] - report_eda['min']
print('Ukuran n,p:', X_raw.shape)
print('Total missing:', int(X_raw.isna().sum().sum()))
print('Duplikat persis:', int(X_raw.duplicated().sum()))
print('Fitur konstan:', list(X_raw.columns[X_raw.nunique() <= 1]))
print('Rasio rentang terbesar/terkecil =', round(report_eda['range'].max()/report_eda['range'].min(),2))
display(report_eda.round(3))

### P2-B. Distribusi, outlier, dan seberapa timpang skala fitur?
**JALANKAN:** tinjau histogram beberapa fitur dan jumlah nilai di luar batas IQR (indikasi statistik, bukan otomatis kesalahan). Tingginya skala `proline` perlu dibandingkan dengan fitur berskala kecil.

In [ ]:
selected = ['alcohol', 'color_intensity', 'proline']
fig, axes = plt.subplots(1,3, figsize=(12,3.2))
for ax,col in zip(axes, selected):
    ax.hist(X_raw[col], bins=18, edgecolor='white')
    ax.set_title(f'Distribusi {col}'); ax.set_xlabel(col); ax.set_ylabel('Jumlah sampel')
plt.tight_layout(); plt.show()
q1, q3 = X_raw.quantile(.25), X_raw.quantile(.75)
iqr = q3-q1
outlier_iqr = ((X_raw.lt(q1-1.5*iqr)) | (X_raw.gt(q3+1.5*iqr))).sum()
print('Jumlah observasi di luar batas 1.5×IQR per fitur (tidak otomatis dihapus):')
display(outlier_iqr.sort_values(ascending=False).rename('jumlah_outlier').to_frame().head(8))

### P2-C. Korelasi, kedekatan, dan tampilan dua dimensi
**JALANKAN:** perhatikan korelasi absolut tertinggi, lalu bandingkan histogram jarak pasangan pada data mentah vs distandardisasi. Angka jarak antarrepresentasi **tidak dapat dibandingkan sebagai jarak bersatuan sama**. Scatter 2D hanyalah proyeksi dua fitur, bukan gambaran lengkap 13 dimensi.

In [ ]:
corr = X_raw.corr(numeric_only=True)
upper = corr.abs().where(np.triu(np.ones(corr.shape), k=1).astype(bool))
flat = upper.stack().sort_values(ascending=False)
print('Pasangan korelasi absolut terbesar:')
display(flat.head(6).rename('abs(Pearson r)').to_frame())
fig, axes=plt.subplots(1,2,figsize=(11,4))
im=axes[0].imshow(corr, vmin=-1,vmax=1,cmap='coolwarm')
axes[0].set_xticks(range(13), labels=X_raw.columns, rotation=90, fontsize=7)
axes[0].set_yticks(range(13), labels=X_raw.columns, fontsize=7)
axes[0].set_title('Korelasi Pearson antar fitur')
plt.colorbar(im,ax=axes[0],fraction=.047)
axes[1].scatter(X_raw['alcohol'],X_raw['color_intensity'],alpha=.68,s=19)
axes[1].set_title('Dua fitur asli, bukan 13D lengkap')
axes[1].set_xlabel('alcohol'); axes[1].set_ylabel('color_intensity')
plt.tight_layout();plt.show()
X_std = StandardScaler().fit_transform(X_raw)
draw, dstd = pdist(X_raw.to_numpy()), pdist(X_std)
fig,axes=plt.subplots(1,2,figsize=(10,3.1))
axes[0].hist(draw,bins=45);axes[0].set_title('Pairwise distance: fitur mentah')
axes[1].hist(dstd,bins=45);axes[1].set_title('Pairwise distance: setelah StandardScaler')
for ax in axes: ax.set_xlabel('Jarak Euclidean');ax.set_ylabel('Pasangan')
plt.tight_layout();plt.show()
print('Median pairwise raw:',round(float(np.median(draw)),3),'| standardized:',round(float(np.median(dstd)),3))

### P2-D. **WAJIB: issue log sebelum model pertama**
Isi **minimal lima temuan numerik**. Contoh yang benar: “Jumlah missing = ..., sehingga imputasi pada dataset asli [perlu/tidak perlu]”. Temuan dapat menyatakan tidak ditemukan masalah; jangan menciptakan masalah yang tidak ada.

| Temuan/isu data | Bukti angka/tabel dari output | Risiko untuk jarak/cluster | Keputusan sementara |
|---|---|---|---|
| 1. n, p, tipe fitur | Ukuran n,p = (178, 13); seluruh 13 kolom bertipe float64 | n kecil, jadi matriks jarak murah dan dendogram masih terbaca. 13 dimensi tidak bisa dilihat langsung | pakai 13 fitur numerik wine.targettidak dipakai sama sekali |
| 2. Missing & duplikasi | total missing= 0, duplikat perisi= 0, fitur konstan= tidak ada | tidak ada risiko pada euclidean atau bobot ganda pda centroid didata asli | imputasi dan dedup tidak diperlukan untuk wine asli(hanya disimulasikan di P5) |
| 3. Rentang & skala | trentang proline= 1402 vs nonflanonoid_phenols= 0,53, rasio= 2645,28, median jarak pasangan 282,172(mentah) vs 5,004(standar) | proline mendominasi jarak euclidean mentah, fitur lain nyari tidak berpengaruh | scaling wajib, Baseline pakai StandaeScaler, RobustScaler diuji sebagai pemanding |
| 4. Outlier berdasarkan IQR | Total 21 nilai di luar 1,5×IQR pada 7 fitur. Terbanyak 4 nilai pada magnesium, alcalinity_of_ash, dan color_intensity (4/178 = 2,2%). alcohol = 0 | nilai ekstrim dapat menggeser centroid K-Means dan urutan merge ward, tetapi jumlahnya kecil dan belum tentu salah | tidak dihapus, diuji sebagai analisis sensitivitas |
| 5. Korelasi/redundansi | Korelasi absolut terbesar total_phenols,flavanoids = 0,865, berikutnya flavanoids od280/od315 = 0,787. Hanya 1 pasangan di atas 0,80 | informasi fenol terhitung ganda dalam jarak, tetapi redudansinya masih terbatas | Jangan langsung membuang fitur. Bandingkan ALL vs seleksi korelasi (buang flavanoids) vs PCA90 di P4 |

**✍️ PERTANYAAN KELAYAKAN.** 
(1) Mengapa dendrogram mungkin berguna untuk 178 sampel? 
(2) Apakah skala fitur dapat membelokkan jarak? 
(3) Apakah data harus dipaksa menjadi cluster kompak? 
(4) Kapan centroid K-Means lebih bermanfaat daripada riwayat merge?

Jawab pertanyaan:
1. Ukurannya cukup kecil karna matriks jarak penuh hanya sekitar 0,2 MB, sehingga seluruh riwayat merge dapat dihitung dan dibaca. Tinggi tiap merge menunjukkan seberapa “mahal” penggabungan dua kelompok. Dari situ k bisa dipilih tanpa label, dan hubungan antar-kelompok terlihat

2. Ya, dan datanya membuktikannya. Median jarak pasangan 282,172 pada data mentah vs 5,004 setelah standardisasi. Proline menyumbang sekitar 99,8% varians total, sehingga jarak mentah praktis hanyalah jarak proline

3. Tidak, EDA hanya menunjukan data layak diuji(bersih dan numerik, tetapi skalanya timpang). EDA tidak menunjukan bahwa cluster kompak pasi ada. Kualitas partisi baru bisa dinilai lewat ekspersimen sebanding di P3, dan hasila sedang saja(silhouette sekitar 0,28%), jadi cluster tida boleh dianggap kelompok alami yang tegas

4. Centroid lebih bermanfaat jika,
- yang dibutuhkan adalah prototipe atau profile tiap segmen( rata-rata fitur per cluster)
- k sudah ditetapkan
- data baru perlu ditugaskan ke segemn lewat perdict
- n besar sehingga matriks jarak mahal
jadi, riwayat merge lebih bermanfaat jika k belum diketahui dan ingin melihat struktur bertingkat


**Kesimpulan EDA sementara (sebelum model):** `
kesimpulan sementara, jika wine asli bersih (0 missing, 0 duplikat, 178x13, semua numerik), tetapi skalanya sangat timpang (rasio rentang 2645,28), sehingga scaling wajib sebelum memakai jarak. Outlier IQR sedikit (21 nilai) dan hanya satu pasangan fitur berkorelasi di atas 0,80, jadi preprocessing cukup ringan. Karena n kecil dan tujuannya eksploratif, Ward layak diuji. K-Means juga layak karena tujuannya segmentasi`

## P3 — Bandingkan algoritma dengan kondisi yang **sama**

**Aturan eksperimen adil:** Ward dan K-Means dibandingkan pada **X dan k yang sama**. Baseline pertama memakai `StandardScaler`, Euclidean, `k=2–5`; K-Means `n_init=20`, `random_state=2026`. `silhouette` lebih tinggi biasanya lebih baik; `DBI` lebih kecil biasanya lebih baik, tetapi keduanya tidak membuktikan makna domain.

**Konstruk kode evaluasi** (jalankan, tidak perlu mengubah kecuali diminta): `evaluate_run()` menyimpan identitas run, model, representasi, k, skor pada **ruang evaluasi standar X_std yang sama**, skor pada ruang training, DBI, ukuran cluster terkecil, dan waktu fit. Score pada ruang training berbeda hanya bersifat deskriptif, **bukan dasar membandingkan raw dengan PCA**.

In [ ]:
EXPERIMENTS = []
ALL_LABELS = {}

def evaluate_run(run_id, X_fit, method='Ward', k=3, X_eval=None, notes=''):
    X_fit = np.asarray(X_fit, dtype=float)
    X_eval = X_fit if X_eval is None else np.asarray(X_eval, dtype=float)
    assert len(X_fit)==len(X_eval), 'Jumlah baris ruang fit dan evaluasi harus sama.'
    t0 = time.perf_counter()
    if method == 'Ward':
        est = AgglomerativeClustering(n_clusters=k, linkage='ward')
    elif method == 'K-Means':
        est = KMeans(n_clusters=k, init='k-means++', n_init=20, random_state=SEED)
    else: raise ValueError('Pilih Ward atau K-Means')
    labs = est.fit_predict(X_fit)
    elapsed = time.perf_counter()-t0
    sizes = np.bincount(labs)
    sil_eval = silhouette_score(X_eval,labs) if 1<len(sizes)<len(labs) else np.nan
    dbi_eval = davies_bouldin_score(X_eval,labs) if 1<len(sizes)<len(labs) else np.nan
    sil_fit = silhouette_score(X_fit,labs) if 1<len(sizes)<len(labs) else np.nan
    # SSE/within-cluster sum of squares, dihitung untuk KEDUA metode di X_eval yang sama.
    # Hanya layak dibandingkan untuk n, k, dan ruang evaluasi yang sama.
    sse = sum(((X_eval[labs==j]-X_eval[labs==j].mean(axis=0))**2).sum()
              for j in np.unique(labs))
    row = {'run':run_id,'model':method,'k':k,'n':len(labs),'fit_dim':X_fit.shape[1],
           'silhouette_common':sil_eval,'DBI_common':dbi_eval, 'SSE_common':sse,
           'silhouette_fit_space':sil_fit,
           'min_size':int(sizes.min()),'max_size':int(sizes.max()),
           'size_pattern':str(sorted(sizes.tolist())), 'fit_seconds':elapsed,'catatan':notes}
    EXPERIMENTS.append(row)
    ALL_LABELS[run_id]=labs
    return row

print('Fungsi perbandingan siap. Nama metrik utama: silhouette_common dan DBI_common.')

### P3-A. Baseline B1: Ward vs K-Means, data dan k sama
**JALANKAN:** 8 run (2 algoritma × k=2,3,4,5) pada Wine standar. 

**JAWAB:** apakah silhouette, DBI, dan ukuran cluster menunjuk metode yang sama? Apakah Ward punya manfaat non-numerik (dendrogram)?

Jawab:
- Sebagian saja, salah satunya k=3 dan k=5, silhouette dan DBI sama-sama menunjuk K-Means. Pada k=3 selisihnya kecil (0,285 vs 0,277 dan 1,389 vs 1,419), sedangkan pada k=5 lebih jelas (0,237 vs 0,187 dan 1,644 vs 1,923)

- Ya, dendrogram menampilkan seluruh riwayat penggabungan, sehingga partisi k=2 dan k=3 saling bersarang (k=2 adalah satu merge dari k=3). Partisi K-Means pada k berbeda tidak otomatis bersarang

In [ ]:
for k in [2,3,4,5]:
    for method in ['Ward','K-Means']:
        evaluate_run(f'B1_{method.replace("-","")}_k{k}',X_std,method,k,X_eval=X_std,notes='Wine standardized, 13 fitur')
baseline_table=pd.DataFrame(EXPERIMENTS)
display(baseline_table[['run','model','k','silhouette_common','DBI_common','min_size','max_size']].round(3))
fig,ax=plt.subplots(figsize=(8.2,3.7))
for method,sub in baseline_table.groupby('model'):
    sub=sub.sort_values('k');ax.plot(sub.k,sub.silhouette_common,marker='o',label=method)
ax.set(title='Wine 13 fitur: silhouette pada ruang StandardScaler yang sama',xlabel='Jumlah cluster (k)',ylabel='Silhouette')
ax.legend();ax.grid(alpha=.2);plt.tight_layout();plt.show()

### P3-B. Dendrogram: kapan Hierarchical punya nilai tambah?
**JALANKAN:** dendrogram Wine standardized. Untuk keterbacaan, grafik menggunakan `truncate_mode='lastp'`; **semua 178 sampel tetap dipakai** untuk membentuk linkage. Cabang pada tampilan terpotong adalah agregasi, bukan observasi asli.

In [ ]:
Z_ward = linkage(X_std, method='ward', metric='euclidean')
fig,ax=plt.subplots(figsize=(11,4))
dendrogram(Z_ward,truncate_mode='lastp',p=20,show_leaf_counts=True,ax=ax)
ax.set(title='Dendrogram Ward — Wine standardized (20 node akhir ditampilkan)',xlabel='Daun/cabang gabungan',ylabel='Tinggi linkage Ward')
plt.tight_layout();plt.show()

**✍️ JAWABAN ANDA — P3 (minimal 100 kata).** Pilih satu k yang menurut Anda layak (bukan otomatis yang skor maksimum). Sebutkan **dua angka** pendukung dan **satu alasan kegunaan dendrogram/centroid**. Jelaskan apakah hasil audit P2 sudah cukup untuk memastikan algoritma terbaik.

**Jawab di sini:** `
Dua angka pendukung.

- Silhouette puncak di k=3 untuk kedua metode (K-Means 0,285 dan Ward 0,277), lalu turun di k=4 (0,261 dan 0,226) dan k=5 (0,237 dan 0,187). Karena Ward k=2 juga cukup dekat (0,267), angka ini saja belum meyakinkan.
- Lompatan tinggi merge pada dendrogram Ward sebesar 15,08. Tinggi merge terakhir 35,40 dan 27,65, lalu turun ke 12,57. Memotong pada tiga cabang menghindari penggabungan yang jauh lebih mahal, sedangkan merge di bawahnya hanya berbeda sekitar 0,3 sampai 1,0.

Kegunaan dendrogram dan centroid. Dendrogram membantu memilih k tanpa label dengan membaca lompatan tinggi merge, dan menunjukkan bahwa partisi k=2 dan k=3 saling bersarang. Centroid K-Means memberi prototipe tiap segmen sehingga profil cluster dapat dibaca dalam satuan fitur asli (dipakai di P7-B). Kedua metode pada k=3 mirip tetapi tidak identik (ARI 0,853).

Apakah audit P2 sudah cukup memastikan algoritma terbaik? Belum. P2 hanya menjawab kelayakan. Hasilnya menunjukkan data bersih, numerik, dan berskala timpang sehingga perlu scaling, serta n yang kecil membuat Ward layak diuji. P2 tidak mengukur kualitas partisi. Selisih silhouette Ward dan K-Means hanya 0,008, dan nilai sekitar 0,28 tergolong sedang. Pemilihan algoritma karena itu perlu ditopang eksperimen yang sebanding (P3–P4), stabilitas hasil, dan kebutuhan pengguna, bukan EDA saja.`

## P4 — Ablation: uji pengaruh materi preprocessing dan feature engineering P1–P7

**Aturan satu-perubahan:** bandingkan satu keputusan yang diubah, dengan metode dan k tetap. Gunakan `k=3` sebagai titik awal **untuk belajar mekanisme**, bukan klaim jumlah cluster optimal.

Untuk membandingkan representasi berbeda, seluruh partisi dievaluasi pada `X_std` yang sama (`silhouette_common`, `DBI_common`). Dengan ini, kita menilai apakah partisi tetap masuk akal dalam ruang 13 fitur standar. Tampilkan juga skor ruang fit untuk mempelajari sensitivitas metrik.

### P4-A. Ablation scaling: raw vs standard vs robust, Ward dan K-Means
**JALANKAN:** dataset sama; yang berubah hanya cara scaling. Jelaskan mengapa silhouette di ruang mentah bisa tinggi meskipun fitur berunit besar mendominasi.

In [ ]:
X_robust = RobustScaler().fit_transform(X_raw)
for method in ['Ward','K-Means']:
    for tag,Xv in [('RAW',X_raw.to_numpy()),('STD',X_std),('ROBUST',X_robust)]:
        evaluate_run(f'P4_{method.replace("-","")}_{tag}', Xv, method, 3, X_eval=X_std,
                     notes=f'pengaruh scaling: {tag}')
rows_scaling = pd.DataFrame(EXPERIMENTS).query("run.str.startswith('P4_')",engine='python')
display(rows_scaling[['run','silhouette_common','silhouette_fit_space','DBI_common','min_size']].round(3))

### P4-B. Ablation *feature selection*: menghapus fitur sangat berkorelasi
**JALANKAN:** fitur diseleksi dengan filter korelasi absolut `>0.80`; angka ambang dipilih sebagai contoh, **bukan aturan universal**. Dalam clustering, fitur dengan korelasi tinggi bisa tetap punya arti domain; selalu catat fitur yang dibuang.

In [ ]:
THRESHOLD_R = 0.80
corr_abs = X_raw.corr().abs()
upper_only = corr_abs.where(np.triu(np.ones(corr_abs.shape),k=1).astype(bool))
to_drop = [c for c in upper_only.columns if (upper_only[c] > THRESHOLD_R).any()]
cols_keep = [c for c in X_raw.columns if c not in to_drop]
X_select = StandardScaler().fit_transform(X_raw[cols_keep])
print('Fitur dibuang:', to_drop, '| jumlah tersisa:', len(cols_keep), 'dari 13')
for method in ['Ward','K-Means']:
    evaluate_run(f'P4_{method.replace("-","")}_SELECT', X_select,method,3,X_eval=X_std,notes='Corr filter > .80')

### P4-C. Ablation PCA 90% explained variance
**JALANKAN:** standardisasi dilanjutkan PCA; cek berapa komponen yang diperlukan untuk mencapai minimal 90% variance. PCA adalah transformasi gabungan fitur, bukan pemilihan kolom asli; skor ruang PCA tidak sepenuhnya sebanding dengan skor ruang standar, maka gunakan *common evaluation space*.

In [ ]:
pca = PCA(n_components=0.90, svd_solver='full')
X_pca = pca.fit_transform(X_std)
print('PCA: dimensi', X_std.shape[1], '→', X_pca.shape[1],
      '| retained cumulative EVR:',round(float(pca.explained_variance_ratio_.sum()),4))
for method in ['Ward','K-Means']:
    evaluate_run(f'P4_{method.replace("-","")}_PCA90', X_pca, method,3,X_eval=X_std,notes='StandardScaler + PCA 90%')
fig,ax=plt.subplots(figsize=(6,2.9))
ax.plot(range(1,X_pca.shape[1]+1),np.cumsum(pca.explained_variance_ratio_),marker='o')
ax.axhline(.90,ls='--',label='Target EVR 90%')
ax.set(title='PCA: informasi variance yang dipertahankan',xlabel='Jumlah komponen',ylabel='Cumulative explained variance')
ax.set_ylim(0,1.03);ax.legend();plt.tight_layout();plt.show()

### P4-D. Ablation nilai ekstrem: clip berbasis IQR (analisis sensitivitas)
**JALANKAN:** membatasi nilai di luar 1,5×IQR sebagai **eksperimen sensitivitas**, bukan keputusan otomatis bahwa nilai ekstrem salah. Jangan mengklaim setiap perubahan skor adalah perbaikan yang sah tanpa alasan domain.

In [ ]:
low, high = q1-1.5*iqr, q3+1.5*iqr
X_clip = X_raw.clip(lower=low,upper=high,axis='columns')
changed_cells = int((X_clip != X_raw).sum().sum())
X_clip_std=StandardScaler().fit_transform(X_clip)
print('Nilai sel yang berubah akibat IQR clipping:',changed_cells)
for method in ['Ward','K-Means']:
    evaluate_run(f'P4_{method.replace("-","")}_CLIP', X_clip_std,method,3,X_eval=X_std,notes='IQR clip pada fitur asli')

### P4-E. Tabel percobaan terkendali
**JALANKAN:** bandingkan **B1 k=3** dengan P4 scaling, selection, PCA, dan clipping. 

**JAWAB:** fitur/dimensi apa yang berubah? Apakah nilai *common* dan *fit* bergerak searah? Mengapa “skor lebih tinggi” tidak otomatis berarti metode lebih baik?

Jawab:
Dari semua ablation, hanya scaling yang benar-benar mengubah hasil, silhouette di ruang yang sama turun dari 0,277 (Ward) dan 0,285 (K-Means) menjadi 0,106 dan 0,108 tanpa scaling. Seleksi fitur, PCA90, dan clipping IQR hanya mengubah skor sekitar 0,01 atau kurang, jadi tidak memperbaiki model secara nyata. Konfigurasi paling sederhana (StandardScaler, 13 fitur, tanpa clipping) dipertahankan. Skor ruang fit dan ruang common tidak selalu searah. Pada data mentah skor fit naik (0,564) padahal skor common turun, dan pada PCA90 skor fit naik padahal skor common tetap. Karena itu skor yang lebih tinggi tidak otomatis berarti model lebih baik. Perbandingan harus dilakukan di ruang evaluasi yang sama, dengan satu perubahan per run, dan tetap memperhitungkan informasi yang hilang serta makna domain.

In [ ]:
all_results = pd.DataFrame(EXPERIMENTS)
compare_p4=all_results[(all_results.k.eq(3)) & ((all_results.run.str.startswith('B1_')) | (all_results.run.str.startswith('P4_')))]
display(compare_p4[['run','model','fit_dim','silhouette_common','DBI_common',
                    'silhouette_fit_space','min_size','size_pattern']].round(3))

**✍️ JAWABAN ANDA — P4.** Isi **minimal tiga** pasangan ablation dan satu kalimat sebab-akibat yang masuk akal. Klaim “PCA memperbaiki model karena skor naik” belum cukup; jelaskan pengurangan dimensi dan risiko hilangnya informasi.

| Dua run dibandingkan | Hanya keputusan apa yang berubah? | Perubahan metrik (angka) | Penjelasan + trade-off |
|---|---|---|---|
| B1_*_k3 vs P4_*_RAW | Scaling: StandardScaler diganti tanpa scaling | Silhouette common Ward 0,277 → 0,106 dan K-Means 0,285 → 0,108. DBI Ward 1,419 → 3,427 dan K-Means 1,389 → 3,364. Silhouette fit justru naik: 0,277 → 0,564 (Ward), 0,285 → 0,571 (K-Means) | proline menyumbang sekitar 99,8% varians mentah, sehingga clustering hampir satu dimensi dan skor di ruang fit tampak tinggi. Di ruang standar yang sama partisinya buruk (ARI vs STD 0,342 dan 0,354). Skor fit dan common bergerak berlawanan |
| P4_*_STD vs P4_*_ROBUST | Jenis scaler: StandardScaler → RobustScaler | Ward 0,277 → 0,275, DBI 1,419 → 1,389. K-Means 0,285 → 0,280, DBI 1,389 → 1,391 | Selisihnya kecil karena outlier IQR hanya 21 sel. Dampak besar datang dari adanya scaling, bukan jenis scalernya. |
| B1_*_k3 (13 fitur) vs P4_*_SELECT (12 fitur) | Membuang flavanoids (korelasi 0,865 dengan total_phenols) | Ward 0,277 → 0,270, DBI 1,419 → 1,407. K-Means 0,285 → 0,284, DBI 1,389 → 1,394. ARI vs ALL 0,885 (Ward) dan 0,947 (K-Means) | Metrik hampir tidak berubah, tetapi sebagian anggota cluster Ward berpindah. Redundansi turun, namun flavanoids punya arti domain dan ikut membentuk profil cluster |

**Jawab di sini:** `
Fitur atau dimensi yang berubah: RAW, STD, dan ROBUST sama-sama 13 fitur (hanya skalanya berbeda), SELECT 12 fitur, PCA90 8 komponen, dan CLIP 13 fitur dengan 21 sel dibatasi. Pada RAW, skor ruang fit (0,564 dan 0,571) dan skor common (0,106 dan 0,108) bergerak berlawanan. Pada PCA90 skor fit naik (0,305 dan 0,315) sedangkan skor common praktis tetap. Pada ROBUST, SELECT, dan CLIP keduanya bergerak searah. Selisih di ruang common kecil. Di ruang fit selisih ROBUST dan SELECT sekitar 0,02, wajar karena ruangnya sendiri berbeda.`

## P5 — Bukti dari P4–P5: apa akibat **missing, duplikasi, dan outlier**?

Wine asli tidak memiliki missing/duplikasi persis. Karena itu kita membentuk **SALINAN TERGANGGU secara sengaja**. Baris asli tidak diubah; simulasi ini **bukan karakteristik Wine asli**. Tujuannya menunjukkan prosedur cleaning dan kualitas data memengaruhi hasil clustering.

**Keputusan penting:** model clustering memerlukan matriks numerik tanpa NaN. Nilai yang hilang harus ditangani; duplikasi dapat mengubah bobot pola; outlier dapat menggeser centroid atau mengubah urutan merge. Jangan menilai preprocessing yang dipilih berdasarkan label asli Wine.

In [ ]:
X_dirty = X_raw.copy()
# Sengaja menaruh missing pada beberapa sel, menduplikasi beberapa baris,
# dan membuat 1 nilai proline ekstrem; semua ini simulasi pembelajaran.
missing_rng = np.random.default_rng(SEED+11)
missing_pairs = missing_rng.choice(len(X_dirty)*3, size=20, replace=False)
for idx in missing_pairs:
    r, cidx = divmod(int(idx), 3)
    X_dirty.iloc[r, cidx] = np.nan
outlier_row=17
original_proline=float(X_dirty.iloc[outlier_row]["proline"])
X_dirty.loc[outlier_row, 'proline'] = original_proline*8
X_dirty=pd.concat([X_dirty, X_dirty.loc[[0,2,59,60]]],ignore_index=True)
print('Wine ASLI:',X_raw.shape, 'missing',int(X_raw.isna().sum().sum()),'duplikat',int(X_raw.duplicated().sum()))
print('Wine SALINAN TERGANGGU:',X_dirty.shape,'missing',int(X_dirty.isna().sum().sum()),
      'duplikat',int(X_dirty.duplicated().sum()))
print('Outlier buatan: proline row 17 sebelum',original_proline,'sesudah',original_proline*8)

### P5-A. Bandingkan dua penanganan: naive vs audit + cleaning
**PENTING:** `naive` memakai imputasi mean dan tetap membiarkan duplikasi serta outlier buatan. `cleaned` memakai median, menghapus duplikasi persis sebelum fit scaler, lalu clipping ringan dari kuantil 1%–99% sebagai **contoh kebijakan simulasi**, bukan aturan baku. Karena penghapusan duplikasi mengubah n, metrik antar-output tidak apple-to-apple. Untuk **bukti perbandingan adil**, model fit pada versi yang sudah dibersihkan lalu **diproyeksikan kembali pada 178 baris bersih asli** tidak trivial untuk Ward yang tidak punya `predict`. Karena itu bagian ini fokus pada perubahan hasil, jumlah anggota, dan audit kualitas; **jangan membandingkan angka antar-n secara langsung untuk menyatakan menang**.

In [ ]:
X_dirty_naive=SimpleImputer(strategy='mean').fit_transform(X_dirty)
X_dirty_naive_std=StandardScaler().fit_transform(X_dirty_naive)
X_dirty_dedup=X_dirty.drop_duplicates().copy()
X_dirty_median=pd.DataFrame(SimpleImputer(strategy='median').fit_transform(X_dirty_dedup),columns=X_raw.columns)
ql=X_dirty_median.quantile(.01); qh=X_dirty_median.quantile(.99)
X_dirty_clean=X_dirty_median.clip(lower=ql,upper=qh,axis='columns')
X_dirty_clean_std=StandardScaler().fit_transform(X_dirty_clean)
for tag,Xv in [('DIRTY_MEAN_NO_CLEAN',X_dirty_naive_std),('CLEAN_MEDIAN_CLIP',X_dirty_clean_std)]:
    for method in ['Ward','K-Means']:
        evaluate_run(f'P5_{method.replace("-","")}_{tag}',Xv,method,3,X_eval=Xv,
                     notes='DATA SIMULASI TERGANGGU; n bisa berubah')
P5_table=pd.DataFrame(EXPERIMENTS)
display(P5_table[P5_table.run.str.startswith('P5_')][['run','n','silhouette_fit_space','DBI_common','min_size','max_size']].round(3))

### P5-B. **Ablation yang lebih adil:** satu masalah data dalam satu waktu, jumlah baris tetap 178

**JALANKAN:** supaya perubahan bisa diatribusikan lebih jelas, buat tiga dataset dengan baris sama: (i) **original**, (ii) **missing 20 sel lalu median imputation**, (iii) **satu outlier buatan pada `proline`**. Bandingkan partisi pada `X_std` asli yang **sama**. Ini mengisolasi dampak yang lebih spesifik daripada P5-A.

In [ ]:
X_miss_only=X_raw.copy()
for idx in missing_pairs:
    r,cidx=divmod(int(idx),3)
    X_miss_only.iloc[r,cidx]=np.nan
X_miss_imp=SimpleImputer(strategy='median').fit_transform(X_miss_only)
X_miss_std=StandardScaler().fit_transform(X_miss_imp)
X_outlier_only=X_raw.copy()
X_outlier_only.loc[outlier_row,'proline']=original_proline*8
X_outlier_std=StandardScaler().fit_transform(X_outlier_only)
for tag,arr in [('ORIGINAL',X_std),('MISSING_MEDIAN',X_miss_std),('OUTLIER_X8',X_outlier_std)]:
    for method in ['Ward','K-Means']:
        evaluate_run(f'P5_ISOLATED_{method.replace("-","")}_{tag}',arr,method,3,X_eval=X_std,
                     notes='n=178, data simulasi, evaluasi X_std original')
isolated=pd.DataFrame(EXPERIMENTS)
display(isolated[isolated.run.str.startswith('P5_ISOLATED')][['run','n','silhouette_common',
           'DBI_common','SSE_common','min_size']].round(3))

**✍️ JAWABAN ANDA — P5.** Mengapa missing harus ditangani sebelum Euclidean? Mengapa duplikasi dapat memengaruhi pusat K-Means? Apakah nilai sangat besar pasti salah? Mengapa angka skor dari `n` berbeda **tidak dapat** dipakai membuktikan hasil cleaning lebih baik? Berikan sedikitnya satu bukti angka dari output.

**Jawab di sini:** `
Jarak Euclidean menjumlahkan selisih kuadrat di semua koordinat. Satu nilai NaN membuat jarak tidak terdefinisi, dan scikit-learn menolak input yang berisi NaN. Missing tidak bisa diabaikan begitu saja. Nilainya harus diimputasi (di sini dengan median, yang kurang sensitif terhadap nilai ekstrem) atau barisnya dibuang dengan pertimbangan. Dan mengapa dapat memengarahui pusat K-means? Baris yang digandakan dihitung dua kali, sehingga centroid tertarik ke arahnya. Ward juga memakai ukuran cluster dalam ΔSSE. Pola yang bukan pengamatan independen jadi mendapat bobot lebih tanpa alasan. Pada simulasi, 4 baris (indeks 0, 2, 59, 60) digandakan sehingga n menjadi 182 dan duplikat persis = 4. Apakah nilai yang paling besar pasti salah? Tidak. Itu hanya indikasi statistik. Dalam data nyata, nilai setinggi itu bisa berupa kesalahan satuan atau pengukuran, tetapi bisa juga sampel yang memang langka, jadi perlu dicek dengan pengetahuan domain sebelum dihapus atau di-clip.

Mengapa skor n berbeda tidak dapat membuktikan  cleaning lebih baik? Pada P5-A data kotor berukuran n=182 dan data bersih n=178. Barisnya berbeda (4 duplikat dibuang), nilainya berbeda (imputasi median dan clipping kuantil 1%–99%), dan tiap skor dihitung di ruang standar masing-masing. Populasi yang dinilai tidak sama, jadi bukan perbandingan apel dengan apel.

Bukti angka: silhouette ruang fit Ward 0,252 (kotor, n=182) vs 0,280 (bersih, n=178), dan K-Means 0,261 vs 0,289. Angka ini hanya deskriptif dan tidak membuktikan bahwa cleaning memperbaiki hasil. Perbandingan yang sah adalah P5-B: satu masalah per run, n tetap 178, dan evaluasi di X_std asli yang sama.`

## P6 — Dua kondisi data: kapan Ward, kapan K-Means?

### Kasus A — Wine asli, 178 × 13, tujuan memahami struktur bertingkat

**EDA berkata:** dataset relatif terkelola, fitur numerik, skala sangat berbeda, ada beberapa korelasi tinggi, dan tujuan *eksplorasi hierarki* dapat dibantu dendrogram. Karena itu **Ward layak diuji**. Namun jika yang dicari hanya segmen kompak, K-Means juga layak. Tabel P3 menentukan apakah ada trade-off kualitas numerik dengan nilai dendrogram.

### Kasus B — 3.000 titik **SINTETIS**, kelompok kompak, tujuan segmentasi praktis

Dataset di bawah **bukan data riil** dan tidak digunakan untuk klaim tentang Wine. Fungsi `make_blobs` hanya menghasilkan dua fitur numerik dan kelompok kompak supaya mudah memahami apakah metode centroid memiliki alasan desain yang kuat. Parameter `centers` diketahui oleh pembuat contoh, tetapi label generator **tidak** dimasukkan ke model.

**JALANKAN:** EDA singkat data besar, uji K-Means semua baris, perkirakan pertumbuhan kebutuhan memori matriks jarak. Untuk benchmark adil kualitas K-Means vs Ward, gunakan **subsample yang sama** (`n=450`) dan k yang sama. Keputusan skala besar berlandaskan kebutuhan proses, biaya, dan sifat data—bukan klaim bahwa Ward tidak bisa dijalankan.

In [ ]:
X_big, _labels_generator_dilarang = make_blobs(
    n_samples=3000, centers=[[-5,-4],[1,4],[7,-1]], cluster_std=[1.15,1.2,1.4],
    random_state=SEED, n_features=2)
# Label generator tidak digunakan untuk fit atau memilih hasil.
big = pd.DataFrame(X_big,columns=['fitur_x','fitur_y'])
print('EDA data sintetis:',big.shape, 'missing:',big.isna().sum().sum(),
      'duplicate:',big.duplicated().sum())
display(big.agg(['min','median','std','max']).round(2))
fig,ax=plt.subplots(figsize=(6.4,3.5))
ax.scatter(big.fitur_x,big.fitur_y,s=4,alpha=.4)
ax.set(title='Kasus B: 3.000 titik sintetis tanpa label',xlabel='fitur_x',ylabel='fitur_y');plt.tight_layout();plt.show()
for n in [178, 450, 3000, 10000]:
    full_mb=(n*n*8)/(1024*1024)
    condensed_mb=(n*(n-1)/2*8)/(1024*1024)
    print(f'n={n:5d} | n×n float64 kira-kira {full_mb:7.1f} MB | condensed distances {condensed_mb:7.1f} MB')
print('Estimasi memori HANYA matriks jarak: bukan total memory atau jaminan alokasi dari library.')

In [ ]:
X_big_std=StandardScaler().fit_transform(big)
# n=450 sama untuk benchmark kualitas (menggunakan random sample index yang pasti)
idx_sampel=np.random.default_rng(SEED).choice(len(X_big_std),size=450,replace=False)
X_sample=X_big_std[idx_sampel]
for method in ['Ward','K-Means']:
    evaluate_run(f'P6_SMALL_{method.replace("-","")}',X_sample,method,3,X_eval=X_sample,
                 notes='perbandingan n=450 data sintetis dengan ruang/k sama')
# n=3000: K-Means dipakai karena tujuan partitioning praktis; bukan mengklaim Ward mustahil
large_record=evaluate_run('P6_LARGE_KMeans',X_big_std,'K-Means',3,X_eval=X_big_std,
               notes='segmen kompak n=3000, k ditentukan; Ward tidak wajib pada data besar')
P6_table=pd.DataFrame(EXPERIMENTS)
display(P6_table[P6_table.run.str.startswith('P6_')][['run','n','silhouette_common',
          'DBI_common','min_size','fit_seconds']].round(3))

**✍️ JAWABAN ANDA — P6.** Isilah tabel keputusan berikut dengan **data dari output**, bukan kesimpulan umum tanpa bukti.

| Aspek | Kasus A: Wine nyata | Kasus B: data sintetis 3.000 titik |
|---|---|---|
| Tujuan analisis | Eksplorasi: memahami struktur kemiripan 178 sampel anggur dan memilih k tanpa label | Segmentasi praktis 3.000 titik tanpa kebutuhan melihat hierarki |
| EDA: jumlah & jenis fitur | 178×13, semua numerik, 0 missing, 0 duplikat, rasio rentang 2645,28 sehingga perlu scaling | 3000×2, numerik, 0 missing, 0 duplikat; std fitur_x 5,07 dan fitur_y 3,54 (skala sebanding, tetap distandardisasi) |
| Bukti struktur/kepraktisan | Struktur sedang: silhouette k=3 hanya 0,285 (K-Means) dan 0,277 (Ward). Lompatan dendrogram 15,08 mendukung k=3 | Struktur sangat jelas: silhouette 0,723, DBI 0,397, ukuran cluster 994, 1000, dan 1006; scatter memperlihatkan tiga gumpalan terpisah |
| Biaya komputasi & manfaat dendrogram | Murah: matriks jarak penuh sekitar 0,2 MB, fit Ward sekitar 0,003 s. Dendrogram berguna untuk memilih k dan melihat hubungan antar-cluster | Matriks jarak penuh n=3000 sekitar 68,7 MB (condensed 34,3 MB); n=10.000 sekitar 762,9 MB (condensed 381,4 MB), tumbuh kuadratik. K-Means pada 3000 baris hanya 0,027 s. Dendrogram tidak diperlukan karena tiga kelompok sudah jelas |
| Nilai hasil pembanding adil | k=3, X_std sama: Ward 0,277 / DBI 1,419 / SSE 1305,0; K-Means 0,285 / 1,389 / 1277,9; ARI antarmetode 0,853 | Subsample n=450 sama, k=3: Ward 0,715 / DBI 0,406 / SSE 92,01; K-Means 0,715 / 0,405 / 91,60; ARI antarmetode 0,986 |
| **Metode pilihan sementara + alasan** | K-Means k=3, dengan dendrogram Ward sebagai bukti pendukung pemilihan k. K-Means sedikit unggul di silhouette, DBI, dan SSE pada k=3, memberi centroid untuk profil, dan sama murahnya. Ward k=3 tetap wajar bila hierarki lebih penting | K-Means. Kualitas Ward dan K-Means hampir identik (0,715 vs 0,715), jadi yang menentukan adalah tujuan (segmen dengan centroid dan penugasan data baru), biaya Ward yang tumbuh kuadratik, dan tidak adanya kebutuhan hierarki |

**Penting:** Pernyataan “K-Means lebih cocok karena dataset lebih besar” **belum memadai**. Tambahkan kebutuhan centroid, tidak memerlukan hierarki, serta trade-off metrik pada kondisi sebanding. Dan jangan menyimpulkan Ward selalu menang untuk data kecil.

### P6-C (opsional) — Ketika keduanya belum tentu cocok
**JALANKAN bila waktu cukup:** data bulan sabit sintetis (*make_moons*). Clustering yang hanya mencari kelompok kompak atau menggabungkan berdasarkan Ward mungkin tidak mengikuti bentuk melengkung. Ini pengantar alasan mempelajari **DBSCAN pada Pertemuan 9**. Jangan memaksakan “hierarchical vs K-Means” jika karakter data mengisyaratkan kebutuhan metode lain.

In [ ]:
X_moon,_ = make_moons(n_samples=350,noise=0.06,random_state=SEED)
fig,axes=plt.subplots(1,3,figsize=(11,3.2))
axes[0].scatter(X_moon[:,0],X_moon[:,1],s=8);axes[0].set_title('Bentuk data tanpa label')
for ax,met in zip(axes[1:],['Ward','K-Means']):
    if met=='Ward': labs=AgglomerativeClustering(n_clusters=2,linkage='ward').fit_predict(X_moon)
    else: labs=KMeans(n_clusters=2,n_init=20,random_state=SEED).fit_predict(X_moon)
    ax.scatter(X_moon[:,0],X_moon[:,1],c=labs,s=8,cmap='coolwarm')
    ax.set_title(f'{met}, k=2')
for ax in axes: ax.set_xlabel('x');ax.set_ylabel('y')
plt.tight_layout();plt.show()

## P7 — Keputusan akhir, interpretasi cluster, dan portofolio

### P7-A. Tentukan konfigurasi final berdasarkan **bukti**
**WAJIB:** Pilih satu dari Ward/K-Means untuk Wine. Ubah nilai konfigurasi jika beralasan. Anda boleh memilih metode yang tidak paling tinggi silhouette-nya asalkan pilihan bermotivasi, transparan, dan mempertimbangkan kebutuhan pengguna.

**EDIT BLOK INI:** pilih kombinasi yang benar-benar Anda ujikan. Opsinya: `scaler = 'StandardScaler' / 'RobustScaler' / 'NONE'`; `representation = 'ALL' / 'CORR' / 'PCA90'`; `clip_iqr = True/False`; model `Ward`/`K-Means`, `k=2–5`. Sel final akan **benar-benar membangun transformasi pilihan Anda**, bukan hanya menuliskan namanya. `PCA90` harus disertai scaling agar variasi tidak dikuasai satuan fitur.

In [ ]:
FINAL_CONFIG = {
    'dataset': 'Wine asli; 178 observasi; 13 fitur numerik; tanpa label',
    'scaler':'StandardScaler',  # EDIT: StandardScaler / RobustScaler / NONE
    'representation':'ALL',   # EDIT: ALL / CORR / PCA90
    'clip_iqr':False,         # EDIT: True atau False
    'method':'Ward',          # EDIT: Ward atau K-Means
    'k':3,                    # EDIT: 2,3,4,5
    'seed': SEED,
    'reason':'ISI ALASAN BERDASARKAN BUKTI EDA DAN P3/P4',  # WAJIB EDIT
}
assert FINAL_CONFIG['method'] in ['Ward','K-Means']
assert FINAL_CONFIG['k'] in [2,3,4,5]
assert FINAL_CONFIG['scaler'] in ['StandardScaler','RobustScaler','NONE']
assert FINAL_CONFIG['representation'] in ['ALL','CORR','PCA90']
base_final=X_raw.copy()
if FINAL_CONFIG['clip_iqr']:
    base_final=base_final.clip(lower=low,upper=high,axis='columns')
if FINAL_CONFIG['representation']=='CORR':
    base_final=base_final[cols_keep]
if FINAL_CONFIG['scaler']=='StandardScaler':
    X_final_fit=StandardScaler().fit_transform(base_final)
elif FINAL_CONFIG['scaler']=='RobustScaler':
    X_final_fit=RobustScaler().fit_transform(base_final)
else:
    X_final_fit=base_final.to_numpy()
if FINAL_CONFIG['representation']=='PCA90':
    assert FINAL_CONFIG['scaler']!='NONE', 'PCA90 perlu scaling pada praktikum ini.'
    X_final_fit=PCA(n_components=.90,svd_solver='full').fit_transform(X_final_fit)
if FINAL_CONFIG['method']=='Ward':
    final_labels=AgglomerativeClustering(n_clusters=FINAL_CONFIG['k'],linkage='ward').fit_predict(X_final_fit)
else:
    final_labels=KMeans(n_clusters=FINAL_CONFIG['k'],n_init=20,random_state=SEED).fit_predict(X_final_fit)
print('Final configuration:',FINAL_CONFIG)
print('Fitur keluaran transformasi:',X_final_fit.shape[1])
print('Silhouette pada ruang fit:',round(silhouette_score(X_final_fit,final_labels),4))
print('Silhouette pada ruang evaluasi original standardized:',round(silhouette_score(X_std,final_labels),4))
print('DBI common:',round(davies_bouldin_score(X_std,final_labels),4))
print('Jumlah anggota:',pd.Series(final_labels).value_counts().sort_index().to_dict())

### P7-B. Profil cluster harus dibaca pada satuan **fitur asli**
**JALANKAN:** rata-rata `alcohol`, `color_intensity`, `proline`, dan fitur lain per cluster. Warna atau nomor cluster tidak mempunyai makna kelas. Jangan memberi label normatif “baik/buruk” tanpa domain.

In [ ]:
profile=X_raw.copy()
profile['cluster']=final_labels
size_profile=profile.groupby('cluster').size().rename('jumlah')
means_profile=profile.groupby('cluster')[['alcohol','color_intensity','proline']].mean()
cluster_profile=pd.concat([size_profile,means_profile],axis=1).round(3)
display(cluster_profile)
fig,ax=plt.subplots(figsize=(6.4,3.7))
for c,g in profile.groupby('cluster'):
    ax.scatter(g.alcohol,g.color_intensity,alpha=.65,s=22,label=f'Cluster {c} (n={len(g)})')
ax.set(title='Profil cluster final pada fitur asli (hanya 2 dari 13 fitur)',xlabel='alcohol',ylabel='color_intensity')
ax.legend(fontsize=8);plt.tight_layout();plt.show()

### P7-C. Export tabel, konfigurasi, dan README untuk portofolio GitHub
**JALANKAN:** empat berkas dibuat otomatis pada folder `P08_output`. Setelah itu lengkapi laporan Markdown pada notebook. **README bawaan masih kerangka**, bukan kesimpulan pribadi; wajib diperbaiki sebelum publikasi.

In [ ]:
import json
res=pd.DataFrame(EXPERIMENTS)
res.to_csv(OUTDIR/'P08_experiment_table.csv',index=False)
cluster_profile.to_csv(OUTDIR/'P08_cluster_profile.csv')
with open(OUTDIR/'P08_final_config.json','w',encoding='utf-8') as fh:
    json.dump(FINAL_CONFIG,fh,ensure_ascii=False,indent=2)
readme = f"""# P08 — Memilih Hierarchical atau K-Means berdasarkan EDA

**Dataset utama:** scikit-learn Wine, 178 sampel × 13 fitur numerik. Label target tidak digunakan.

## Pertanyaan proyek
Bagaimana pilihan scaling, feature selection, PCA, dan konteks pemakaian memengaruhi pemilihan Ward atau K-Means?

## Alur eksperimen
Audit data → baseline fair comparison → ablation → simulasi kualitas data → keputusan akhir → profil cluster.

## Keputusan final (perlu Anda jelaskan kembali dengan angka)
- Metode: {FINAL_CONFIG['method']}
- k: {FINAL_CONFIG['k']}
- Scaling: {FINAL_CONFIG['scaler']}
- Representasi: {FINAL_CONFIG['representation']}
- Clipping IQR: {FINAL_CONFIG['clip_iqr']}
- Silhouette pada ruang StandardScaler: {silhouette_score(X_std,final_labels):.4f}
- DBI pada ruang StandardScaler: {davies_bouldin_score(X_std,final_labels):.4f}
- Alasan dari mahasiswa: **[GANTI dengan argumen berdasar EDA, perbandingan, risiko]**

## Bukti
- Notebook utama `.ipynb` (soal, kode, output, interpretasi)
- `P08_experiment_table.csv`
- `P08_cluster_profile.csv`
- `P08_final_config.json`

## Keterbatasan
[TULISKAN minimal tiga: misalnya sensitivitas skala, pilihan k, interpretasi domain, biaya memori.]

## Reproduksi
Python 3; NumPy; Pandas; SciPy; scikit-learn; Matplotlib. Jalankan notebook dari atas sampai bawah.
"""
(OUTDIR/'README_PORTOFOLIO_P08.md').write_text(readme,encoding='utf-8')
print('Berkas dibuat:')
for f in sorted(OUTDIR.iterdir()):print('-',f)

### ✍️ LAPORAN AKHIR — Tulis langsung di notebook (tidak perlu DOCX terpisah)

**L1. Rumusan masalah (2–3 kalimat):** Apa objek dan tujuan pengelompokan? Mengapa clustering, bukan classification?

`Objek analisis adalah 178 sampel anggur Wine, masing-masing dengan 13 fitur hasil analisis kimia. Tujuannya mengelompokkan sampel yang mirip secara kimiawi tanpa memakai label cultivar, sambil membandingkan Ward dan K-Means dengan data, fitur, dan k yang sama. Ini clustering dan bukan classification karena tidak ada label target yang dipakai untuk melatih atau memilih model. Hasilnya hanya kelompok kemiripan, bukan kelas yang benar.`

**L2. Ringkasan EDA:** kutip minimal **lima temuan angka** dan satu keputusan preprocessing yang mengikuti temuan tersebut.

`Menemukan angka:
1. Ukuran 178x13, semua fitur float64
2. Missing= 0 dan duplikat persis= 0
3. Rasio rentang fitur 2645,28 (proline 1402 vs nonflavanoid_phenols) 282,172(mentah) vs 5,004(standar)
4. 21 nilai diluar 1,5xIQR pada 7 fitur, maksimum 4 per fitur
5. Korelasi tertinggi total_phenols-flavanoids= 0,865, dan hanya satu pasangan di atas 0,80`

**L3. Ablation P1–P7:** sebutkan sedikitnya **tiga eksperimen sah**, perubahan masing-masing, metrik pada ruang evaluasi yang sama, serta pengaruh terhadap interpretasi.

`
Di evaluasi X_std yang sama (k=3)
- Scaling, StandarScaler => tanpa scaling menurunkan silhouette common Ward 0,277 => 0,106 dan K-Means 0,285 => 0,108 (DBI 1,419 => 3,427 dan 1,389 => 3,364). Silhouette ruang fit justru naik ke 0,564 dan 0,571 karena proline mendominasi jarak
- Setelah Fitur, membuang flavanoids (13=>12) mengubah ward 0,277 => 0,270 dan K-Means 0,285 => 0,284, ARI 0,885 dan 0,947. Pengaruhnya hampir tidak ada
- PCA90, 13 => 8 komponen (EVR 0,9202). Ward 0,277 => 0,275 dan K-means 0,285 => 0,285 dengan partisi identik (ARI 1,0). Skor fit naik hanya karena dimensi berkurang
- Clip IQR, 21sel berubah. Ward 0,277 => 0,279 dan K-means tidak berubah`

**L4. Mengapa memilih Ward/K-Means untuk Wine?** Sebutkan k dan minimal dua metrik numerik. Jelaskan peran dendrogram atau centroid, biaya komputasi, dan apakah alternatif berbeda mungkin tetap wajar.

`
- Metrik: Silhouette
K-means: 0,285
Ward: 0,277

- Metrik: DBI
K-means: 1,389
Ward: 1,419

- Metrik: SSE
K-means: 1277,9
Ward: 1305,0

Hasilnya stabil, Dendrogram Ward dipakai sebagai bukti pemilihan k lompatan tinggi merge 15,08 antara merge ke-2 dan ke-3 dari atas mendukung tiga cabang. Centroid dipakai untuk profil cluster dalam satuan asli (P7-B). Biaya komputasi pada n=178 trivial untuk keduanya (sekitar 0,003 s untuk Ward dan 0,022 s untuk K-Means) Ward k=3 tetap hierarki yang dibutuhkan, Ward lebih sesuai.
`

**L5. Dua kondisi data:** bandingkan kesimpulan Wine nyata dengan data besar sintetis. Jelaskan mengapa kondisi data dan tujuan aplikasi, bukan sekadar angka silhouette, berpengaruh pada pilihan.

`
Pada wine nyata strukturnya sedang (silhouette sekitar 0,28) dan tujuannya eksploratif, sehingga Ward dan K-Means setara dan dendrogram berguna untuk memilih k. Pada data sintetis 3.000 titik 2D strukturnya sangat jelas (silhouette 0,723, DBI 0,397), dan pada subsample n=450 kedua metode praktis identik (0,715 vs 0,715, ARI 0,986). Yang membedakan keputusan adalah tujuan dan biaya. Untuk segmentasi praktis dengan centroid dan penugasan data baru, K-Means lebih tepat, sedangkan matriks jarak Ward tumbuh kuadratik (n=3000 sekitar 68,7 MB, n=10.000 sekitar 762,9 MB). Silhouette tinggi pada data sintesis hanya menunjukkan data mudah dipisah bukan bahwa metodenya lebih unggul`

**L6. Analisis keterbatasan & penggunaan bertanggung jawab (min. 3):** misalnya skala, PCA, makna cluster, stabilitas, outlier, keterulangan, pembatasan generalisasi.

`
- Skala: hasil sangat bergantung pada scaling (0,106 vs 0,277), sehingga pilihan scaler harus dicatat.
- PCA dan seleksi fitur: PCA90 membuang sekitar 8% varians dan membuat fitur sulit diinterpretasi. Membuang flavanoids menghilangkan fitur yang bermakna domain.
- Makna cluster: silhouette hanya 0,285, ada 7 sampel bersilhouette negatif, dan cluster 1 paling lemah (rata-rata 0,177). Nomor cluster tidak bermakna kelas dan tidak boleh diberi label baik atau buruk tanpa pakar domain.
- Outlier dan stabilitas: satu outlier ekstrem (proline ×8) menggeser Ward lebih besar (ARI 0,870) daripada K-Means (0,930).
- Simulasi: eksperimen P5 memakai data buatan, bukan karakteristik Wine asli.
- Generalisasi: analisis bersifat deskriptif pada satu dataset tanpa split, sehingga tidak boleh diklaim berlaku untuk data baru.
- Asumsi bentuk: K-Means dan Ward mengasumsikan cluster kompak. Pada make_moons keduanya memotong bentuk sabit secara tidak sesuai, sehingga perlu metode lain seperti DBSCAN.`

**L7. Refleksi profesional (3–5 kalimat):** Apa satu keputusan Anda yang berubah setelah melihat EDA/hasil eksperimen? Bagaimana Anda memverifikasinya?

`
Awalnya saya mengira Ward otomatis lebih cocok karena n hanya 178 dan dendrogram dapat dibaca. Setelah melihat tabel baseline, K-Means ternyata unggul tipis di silhouette, DBI, dan SSE pada k=3 dan stabil (ARI 1,0 pada 10 seed), sehingga keputusan saya berubah ke K-Means dengan dendrogram sebagai alat bantu memilih k. Saya juga semula menganggap silhouette 0,564 pada data mentah menandakan hasil bagus, tetapi setelah dievaluasi di ruang X_std yang sama nilainya hanya 0,106. Verifikasinya adalah ablation satu perubahan per run, evaluasi pada ruang common, dan pengulangan dengan beberapa seed.`

**L8. Tautan portofolio (setelah commit):** https://colab.research.google.com/drive/1MKE5zPmdtRiux3I-zSkTLAI9UqetPYJ0?usp=sharing

## CHECKLIST PENGUMPULAN DAN PENILAIAN

- [ ] Semua cell berjalan dari atas ke bawah (*Restart & Run All*) tanpa error.
- [ ] Setiap bagian P1–P7 dan L1–L8 memiliki jawaban nyata, bukan `ISI...`.
- [ ] P2 memuat sedikitnya lima temuan EDA dengan angka.
- [ ] P3 membandingkan kedua algoritma pada **data dan k yang sama**.
- [ ] P4 menunjukkan sedikitnya tiga ablation yang sah.
- [ ] P5 menyebutkan data bermasalah adalah **simulasi**, bukan Wine asli.
- [ ] P6 membedakan kebutuhan hierarki vs kepraktisan centroid & biaya pada ukuran berbeda.
- [ ] P7 membahas cluster profile pada fitur asli dan batas interpretasi.
- [ ] Berkas ekspor tersedia di `P08_output`; notebook dengan output disimpan.
- [ ] GitHub: unggah notebook final + README yang sudah ditulis ulang. Jangan unggah data berlisensi/privat tanpa izin.

